In [ ]:
!pip install huggingface_hub
!pip install transformers==4.57.6 accelerate==1.12.0 optimum==1.26.0
!pip install peft==0.17.0

In [ ]:
!huggingface-cli login --token "hf_token"

In [5]:
import os
import base64
import json

from huggingface_hub import snapshot_download
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import torch
from pprint import pprint

check_points = {
    "100": "cadc94b822d650b76ce2c374a0b706664d6fa0d7",
    "200": "88adc2d62af2d62bedcf9da04524ba61071513a5",
    "350": "eb251e7f49a1b6d174e501f592d4a9e21b00c5fc",
    "1000": "7b378f17b68cc2afd66cb907083313ea2a0f0fd8",
    "1500": "52a04c2746c54adc0139376a6cf4c4068ca688aa",
    "1700": "918bf033daf5fc32bafa707a168d05b84daf3922"
}





def image_to_base64_data_uri(image_path):

    """Convert image to base64 data URI for APIs"""
    with open(image_path, 'rb') as image_file:
        img_base64 = base64.b64encode(image_file.read()).decode('utf-8')

    # Determine image type from extension
    ext = image_path.lower().split('.')[-1]
    mime_type = f"image/{ext}" if ext != "jpg" else "image/jpeg"

    return f"data:{mime_type};base64,{img_base64}"
    



eval_images = ["07_003.jpg", "07_007.jpg",
"18_001.jpg",
"31_001.jpg", "31_004.jpg",
"47_004.jpg", "47_012.jpg"]


task_1_prompt = """
You are a professional OCR Details Extractor.
Your rule to extract: the page markdown content in addition to the structural_elements of the document.
Extract the final output into a json format.
Do not generate any introduction or conclusion.
""".strip()

task_2_prompt = """
You are a professional OCR Details Extractor.
Your rule to extract the: document_classification, source, physical_properties, official_marks, signatures_authorization, routing_distribution, attachments_references, condition_notes and confidence_quality of the document.
Extract the final output into a json format.
Do not generate any introduction or conclusion.
""".strip()

In [ ]:
model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(model_id)

torch.set_grad_enabled(False)


cached_inputs = {}

for image_name in eval_images:
    sample_image = os.path.join("./eval_images", image_name)

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_to_base64_data_uri(sample_image)},
                {"type": "text", "text": task_1_prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    cached_inputs[image_name] = {
        "inputs": inputs,
        "input_len": input_len
    }

## base model outputs
base_outputs = {}
for idx in range(len(eval_images)):

    with torch.inference_mode():
        generation = model.generate(**cached_inputs[eval_images[idx]]["inputs"], max_new_tokens=1024, do_sample=False)
        generation = generation[0][cached_inputs[eval_images[idx]]["input_len"]:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    base_outputs[eval_images[idx]] = decoded
        

## base model + LoRA outputs
lora_outputs = {}
for checkpoint_num, checkpoint_hash in check_points.items():
    path = snapshot_download(
        repo_id="husammm/V3",
        revision=checkpoint_hash,
        allow_patterns="last-checkpoint/*",
        local_dir=f"./loaded_checkpoints/{checkpoint_num}",
        local_dir_use_symlinks=False
        )
    checkpoint_path = f"./loaded_checkpoints/{checkpoint_num}/last-checkpoint"
    ## Load LoRA
    model.load_adapter(checkpoint_path, 
                       adapter_name="lora",
                       is_trainable=False)
    model.set_adapter("lora")
    lora_outputs[checkpoint_num] = {}
    for idx in range(len(eval_images)):
        
        with torch.inference_mode():
            generation = model.generate(**cached_inputs[eval_images[idx]]["inputs"], max_new_tokens=1024, do_sample=False)
            generation = generation[0][cached_inputs[eval_images[idx]]["input_len"]:]

        decoded = processor.decode(generation, skip_special_tokens=True)
        lora_outputs[checkpoint_num][eval_images[idx]] = decoded
        # pprint(decoded)
    
    model.set_adapter(None)
    model.delete_adapter("lora")
    torch.cuda.empty_cache()

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/base_outputs.jsonl", "w") as dest:
    json.dump(base_outputs, dest, ensure_ascii=False, indent=2)
with open("./outputs/lora_outputs.jsonl", "w") as dest:
    json.dump(lora_outputs, dest, ensure_ascii=False, indent=2)
        

In [9]:
os.makedirs("./outputs", exist_ok=True)
with open("./outputs/base_outputs.jsonl", "w") as dest:
    json.dump(base_outputs, dest, ensure_ascii=False, indent=2)

In [ ]:
len(base_outputs)


In [24]:
lora_outputs = {}

In [31]:


for checkpoint_num, checkpoint_hash in check_points.items():
    print(f"Started checkpoint: {checkpoint_num}")
    if checkpoint_num == "100":
        continue
    if checkpoint_num != "200":
        path = snapshot_download(
            repo_id="husammm/V3",
            revision=checkpoint_hash,
            allow_patterns="last-checkpoint/*",
            local_dir=f"./loaded_checkpoints/{checkpoint_num}",
            local_dir_use_symlinks=False
            )
        checkpoint_path = f"./loaded_checkpoints/{checkpoint_num}/last-checkpoint"
        ## Load LoRA
        model.load_adapter(checkpoint_path, 
                           adapter_name=f"{checkpoint_num}",
                           is_trainable=False)
        model.set_adapter(f"{checkpoint_num}")
    lora_outputs[checkpoint_num] = {}
    for idx in range(len(eval_images)):
        print(f"Started image: {eval_images[idx]}")
        with torch.inference_mode():
            generation = model.generate(**cached_inputs[eval_images[idx]]["inputs"], max_new_tokens=1024, do_sample=False)
            generation = generation[0][cached_inputs[eval_images[idx]]["input_len"]:]

        decoded = processor.decode(generation, skip_special_tokens=True)
        lora_outputs[checkpoint_num][eval_images[idx]] = decoded
        # pprint(decoded)
    
    # model.set_adapter(None)
    model.delete_adapter(f"{checkpoint_num}")
    torch.cuda.empty_cache()

with open("./outputs/lora_outputs.jsonl", "w") as dest:
    json.dump(lora_outputs, dest, ensure_ascii=False, indent=2)

Started checkpoint: 100
Started checkpoint: 200
Started image: 07_003.jpg
Started image: 07_007.jpg
Started image: 18_001.jpg
Started image: 31_001.jpg
Started image: 31_004.jpg
Started image: 47_004.jpg
Started image: 47_012.jpg
Started checkpoint: 350


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.21k [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

trainer_state.json:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

last-checkpoint/training_args.bin:   0%|          | 0.00/6.22k [00:00<?, ?B/s]

last-checkpoint/optimizer.pt:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

Started image: 07_003.jpg
Started image: 07_007.jpg
Started image: 18_001.jpg
Started image: 31_001.jpg
Started image: 31_004.jpg
Started image: 47_004.jpg
Started image: 47_012.jpg


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1030: UserWarning: Adapter 1700 was active which is now deleted. Setting active adapter to lora.
  warnings.warn(


In [27]:
model.delete_adapter(f"{checkpoint_num}")
torch.cuda.empty_cache()

In [29]:
len(lora_outputs["100"])

7